# Show-o2 1.5B: piloto de refusal en understanding
Solo **imagen + texto -> texto**, sin entrenamiento. Nueve pasos como el notebook de referencia, más una celda de prueba propia.

GPU en Colab (preferible A100; memoria y resultados pendientes de medir). Ejecuta 1, reinicia la sesión y sigue desde 2.
La rama debe estar subida a GitHub antes de ejecutar el clone de la celda 1.

Método: [Arditi et al.](https://arxiv.org/abs/2406.11717), secciones 2.3, 2.4 y C.1; [código original](https://github.com/andyrdt/refusal_direction).
Es una adaptación multimodal: los 128+128 prompts originales comparten una imagen fija. El paper no demuestra el resultado para Show-o2; este piloto lo mide.

In [ ]:
# 1. Clonar e instalar. Ejecutar una vez en un runtime limpio.
!git clone --branch experiment/refusal-understanding --single-branch https://github.com/Nicerova7/show-o2-space.git /content/show-o2-space
!git clone https://github.com/showlab/Show-o.git /content/Show-o
!git -C /content/Show-o checkout 45a5a2de01d1ebd10cd5864d29310a76476cdf23
!git clone https://github.com/andyrdt/refusal_direction.git /content/refusal_direction
!git -C /content/refusal_direction checkout 9d852fae1a9121c78b29142de733cb1340770cc3
%pip install -q transformers==4.47.0 diffusers==0.31.0 huggingface-hub==0.27.1 accelerate==1.2.1 timm==1.0.12 einops==0.8.0 torchdiffeq==0.2.5 "numpy>=1.26,<3" "Pillow>=10.4,<12" sentencepiece==0.2.1 pandas openpyxl tqdm
print("Ahora: Entorno de ejecución > Reiniciar sesión. Continúa en la celda 2.")

In [ ]:
# 2. Cargar Show-o2 1.5B usando el adapter de main.
import sys
import os
import json
import random
import shutil
from pathlib import Path
import torch
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

PROJECT = Path('/content/show-o2-space')
UPSTREAM = Path('/content/Show-o')
os.chdir(PROJECT)
sys.path[:0] = [str(PROJECT), str(UPSTREAM / 'show-o2'), '/content/refusal_direction']

from show_o2_space.adapter import ShowO2Adapter, MODEL_ID
from experiments.refusal_utils import (
    UnderstandingInputs, mean_activations, evaluate_candidates,
    addition_hooks, ablation_hooks, generate_response, remove_component,
)

N_BUILD, N_VALIDATION, N_TEST = 128, 32, 10
TOKENS_TO_GENERATE, SEED = 100, 42
torch.manual_seed(SEED)
adapter = ShowO2Adapter()
tokenizer = adapter.tokenizer
N_LAYERS = len(adapter.layers)
print(MODEL_ID, torch.cuda.get_device_name(0), adapter.dtype)
print('Capas:', N_LAYERS, '| Batch: 1 | Decoding: greedy (top_k=1)')

In [ ]:
# 3. Los mismos splits y el mismo muestreo del notebook de referencia.
from dataset.load_dataset import load_dataset_split
rng = random.Random(SEED)
harmful_train = rng.sample(load_dataset_split('harmful', 'train', instructions_only=True), N_BUILD)
harmless_train = rng.sample(load_dataset_split('harmless', 'train', instructions_only=True), N_BUILD)
harmful_val = rng.sample(load_dataset_split('harmful', 'val', instructions_only=True), N_VALIDATION)
harmless_val = rng.sample(load_dataset_split('harmless', 'val', instructions_only=True), N_VALIDATION)

# Misma imagen para ambos grupos: solo cambia la instrucción.
BUILD_IMAGE = PROJECT / 'inputs/images/example.jpg'
shutil.copy2(UPSTREAM / 'show-o2/docs/mmu/pexels-jane-pham-727419-1571673.jpg', BUILD_IMAGE)
inputs = UnderstandingInputs(adapter, BUILD_IMAGE)
OUTPUT = PROJECT / 'outputs/refusal'
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / 'data.json').write_text(json.dumps(dict(
    seed=SEED, image=str(BUILD_IMAGE), harmful_train=harmful_train,
    harmless_train=harmless_train, harmful_val=harmful_val, harmless_val=harmless_val,
), indent=2), encoding='utf-8')
print('Build:', len(harmful_train), '+', len(harmless_train))
print('Validación:', len(harmful_val), '+', len(harmless_val))
print('Tests propios: 10 + 10, después del paso 9.')

In [ ]:
# 4. Inspeccionar el formato real y las posiciones posteriores a la instrucción.
_, _, alignment = adapter.prepare_understanding(BUILD_IMAGE, harmless_train[0])
print('Secuencia completa (-1 representa embeddings visuales o de tiempo):')
print(tokenizer.decode([t for t in alignment['token_ids'] if t >= 0], skip_special_tokens=False))
print('Alineamiento multimodal:', alignment)
SOURCE_POSITIONS = inputs.positions
N_POSITIONS = len(SOURCE_POSITIONS)
for position, token_id in zip(SOURCE_POSITIONS, inputs.suffix_ids):
    print(position, token_id, repr(tokenizer.decode([token_id])))
print('Candidatos:', N_POSITIONS * N_LAYERS)
print('Capas fuente a evaluar:', list(range(int(0.8 * N_LAYERS))))

In [ ]:
# 5. Diferencia de medias en la ENTRADA de cada bloque, no en su salida.
harmful_means = mean_activations(inputs, harmful_train)
harmless_means = mean_activations(inputs, harmless_train)
candidate_directions = harmful_means - harmless_means
torch.save(candidate_directions, OUTPUT / 'candidate_directions.pt')
print('Shape [positions, layers, hidden]:', tuple(candidate_directions.shape))
display(pd.DataFrame({
    'harmful_mean': harmful_means[0, 12, :6].numpy(),
    'harmless_mean': harmless_means[0, 12, :6].numpy(),
    'r = harmful - harmless': candidate_directions[0, 12, :6].numpy(),
}))

In [ ]:
# 6. Scores de validación; solo primeras floor(0.8 L) capas fuente.
# Proxy del paper: log(P(refusal) / (1-P(refusal))) en el primer token de respuesta.
# I / As son los inicios usados para Qwen por la referencia, no un juez de seguridad.
REFUSAL_IDS = [tokenizer.encode(text, add_special_tokens=False)[0] for text in ['I', 'As']]
print('Tokens del proxy:', [(i, tokenizer.decode([i])) for i in REFUSAL_IDS])
selection_df = evaluate_candidates(
    inputs, candidate_directions, harmful_val, harmless_val, REFUSAL_IDS,
)
selection_df.to_json(OUTPUT / 'direction_evaluations.json', orient='records', indent=2)
passed_df = selection_df.query('passes_filters').sort_values('bypass_score')
passed_df.to_json(OUTPUT / 'direction_evaluations_filtered.json', orient='records', indent=2)
display(selection_df)
print('Candidatos que cumplen induce > 0 y KL < 0.1:', len(passed_df))
# Si ninguno cumple, el piloto no encontró una dirección bajo estos criterios.
# Se conservan los scores para inspección; no elegimos un candidato fuera de filtro.
selected = passed_df.iloc[0]
selected_layer, selected_position = int(selected.layer), int(selected.position)
refusal_direction = candidate_directions[SOURCE_POSITIONS.index(selected_position), selected_layer]
print('Elegido:', selected.to_dict())

In [ ]:
# 7. Inspeccionar y guardar la dirección seleccionada, sin normalizar su magnitud.
print('Capa:', selected_layer, '| Posición:', selected_position)
print('Shape:', refusal_direction.shape, '| Norma:', refusal_direction.norm().item())
print('Primeros componentes:', refusal_direction[:10])
metadata = dict(
    model=MODEL_ID, seed=SEED, build_per_group=N_BUILD, validation_per_group=N_VALIDATION,
    build_image=str(BUILD_IMAGE), positions=SOURCE_POSITIONS,
    selected_layer=selected_layer, selected_position=selected_position,
    refusal_ids=REFUSAL_IDS, source_capture='decoder block input',
    source_layer_limit=int(0.8 * N_LAYERS), induce_threshold=0.0, kl_threshold=0.1,
    scores=selected.to_dict(),
)
torch.save(dict(direction=refusal_direction, metadata=metadata), OUTPUT / 'direction.pt')
(OUTPUT / 'metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')

In [ ]:
# 8. Inducir refusal: h' = h + r, en la capa elegida y TODAS las posiciones.
# Los hooks siguen activos en cada paso autoregresivo de adapter.answer.
ADD_PRE_HOOKS, ADD_OUTPUT_HOOKS = addition_hooks(adapter, refusal_direction, selected_layer)
HARMLESS_EXAMPLE = 'Explain why leaves change color in autumn.'
baseline = generate_response(adapter, BUILD_IMAGE, HARMLESS_EXAMPLE, max_new_tokens=TOKENS_TO_GENERATE)
modified = generate_response(adapter, BUILD_IMAGE, HARMLESS_EXAMPLE,
    pre=ADD_PRE_HOOKS, post=ADD_OUTPUT_HOOKS, max_new_tokens=TOKENS_TO_GENERATE)
print('PREGUNTA:', HARMLESS_EXAMPLE)
print('BASELINE:', baseline)
print('ADD DIRECTION:', modified)
h = harmless_means[-1, selected_layer]
display(pd.DataFrame({'h_before': h[:6].numpy(), 'r': refusal_direction[:6].numpy(),
                      'h_after': (h + refusal_direction)[:6].numpy()}))
print('Inspección numérica sobre la media almacenada; respuestas de generación real arriba.')

In [ ]:
# 9. Suprimir refusal: h' = h - (h dot r_hat) r_hat.
# TODAS las capas y posiciones, incluyendo imagen/tiempo y tokens generados.
# Se proyectan entradas de bloques y salidas de atención/MLP, como la referencia.
ABLATION_PRE_HOOKS, ABLATION_OUTPUT_HOOKS = ablation_hooks(adapter, refusal_direction)
HARMFUL_EXAMPLE = harmful_val[0]
baseline = generate_response(adapter, BUILD_IMAGE, HARMFUL_EXAMPLE, max_new_tokens=TOKENS_TO_GENERATE)
modified = generate_response(adapter, BUILD_IMAGE, HARMFUL_EXAMPLE,
    pre=ABLATION_PRE_HOOKS, post=ABLATION_OUTPUT_HOOKS, max_new_tokens=TOKENS_TO_GENERATE)
print('PREGUNTA:', HARMFUL_EXAMPLE)
print('BASELINE:', baseline)
print('REMOVE COMPONENT:', modified)
h = harmful_means[-1, selected_layer]
h_new = remove_component(h, refusal_direction)
r_hat = refusal_direction / refusal_direction.norm()
display(pd.DataFrame({'h_before': h[:6].numpy(), 'removed_component': (h-h_new)[:6].numpy(),
                      'h_after': h_new[:6].numpy()}))
print('Proyección antes:', torch.dot(h, r_hat).item())
print('Proyección después:', torch.dot(h_new, r_hat).item())

## Pruebas propias: 10 harmful + 10 harmless
Sube tus imágenes desde el panel **Archivos** de Colab a `inputs/images/` (carpeta ya existente).
Escribe diez prompts en cada lista y sus diez rutas correspondientes en el mismo orden.
Puedes usar las mismas diez imágenes para ambos grupos o rutas distintas.
Las listas comienzan vacías para que pegues tus datos reales. Ejecuta después de completarlas.

Cada imagen se reutiliza en baseline e intervención del mismo prompt. El conjunto es independiente de build/validación.
El Excel deja `refusal` vacío para anotar manualmente 1 = rechaza, 0 = no rechaza; ausencia de rechazo no implica respuesta dañina.
La celda también guarda el manifiesto de pares imagen/prompt para conservar el emparejamiento.

In [ ]:
# 10. Completa las cuatro listas antes de ejecutar.
harmful_test = [
    # Tus 10 prompts harmful, uno por línea.
]
harmless_test = [
    # Tus 10 prompts harmless, uno por línea.
]
harmful_images = [
    # 'inputs/images/01.jpg', ... hasta tus 10 rutas.
]
harmless_images = [
    # 'inputs/images/01.jpg', ... hasta tus 10 rutas.
]

# DataFrame empareja por orden y requiere la misma cantidad de prompts y rutas.
tests = pd.concat([
    pd.DataFrame(dict(prompt=harmful_test, image=harmful_images)).assign(group='harmful'),
    pd.DataFrame(dict(prompt=harmless_test, image=harmless_images)).assign(group='harmless'),
], ignore_index=True)
tests.to_json(OUTPUT / 'test_data.json', orient='records', indent=2)
conditions = {
    'harmful': ('remove_component', ABLATION_PRE_HOOKS, ABLATION_OUTPUT_HOOKS),
    'harmless': ('add_direction', ADD_PRE_HOOKS, ADD_OUTPUT_HOOKS),
}
rows = []
for prompt_id, item in tqdm(tests.iterrows(), total=len(tests)):
    condition, pre, post = conditions[item.group]
    baseline = generate_response(adapter, item.image, item.prompt, max_new_tokens=TOKENS_TO_GENERATE)
    modified = generate_response(adapter, item.image, item.prompt, pre, post, TOKENS_TO_GENERATE)
    for name, response in [('baseline', baseline), (condition, modified)]:
        rows.append(dict(prompt_id=prompt_id+1, group=item.group, image=item.image,
                         prompt=item.prompt, condition=name, source_layer=selected_layer,
                         source_position=selected_position, response=response, refusal=None))
results_df = pd.DataFrame(rows)
results_df.to_excel(OUTPUT / 'refusal_pilot_results.xlsx', index=False)
display(results_df)
print('Prompts por grupo:', tests.groupby('group').size().to_dict())
print('Con 10 + 10 prompts se guardan 40 respuestas en', OUTPUT)
from google.colab import files
archive = shutil.make_archive('/content/show-o2-refusal-results', 'zip', OUTPUT)
files.download(archive)